<a href="https://colab.research.google.com/github/roberaBelachew/clarity_cpc2_exploration.ipynb/blob/main/clarity_cpc2_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import shutil

disk = shutil.disk_usage("/content")

print(f"Free disk space: {disk.free / 1e9:.1f} GB")

Free disk space: 93.6 GB


In [4]:
# Downloading the dataset
!mkdir -p /content/clarity_cpc2

!wget -c \
  "https://zenodo.org/records/17045970/files/clarity_CPC2_data.v1_1.tgz?download=1" \
  -O /content/clarity_cpc2/clarity_CPC2_data.v1_1.tgz



--2026-10-27 09:20:12--  https://zenodo.org/records/17045970/files/clarity_CPC2_data.v1_1.tgz?download=1
Resolving zenodo.org (zenodo.org)... 188.185.40.132, 137.138.121.78, 188.185.51.60, ...
Connecting to zenodo.org (zenodo.org)|188.185.40.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12096370815 (11G) [application/octet-stream]
Saving to: ‘/content/clarity_cpc2/clarity_CPC2_data.v1_1.tgz’

larity_cpc2/clarity   0%[                    ]   4.38M  1016KB/s    eta 3h 12m ^C


In [6]:
# extracting the metadata files
from pathlib import Path
import tarfile

project_dir = Path("/content/clarity_cpc2")
archive_path = project_dir / "clarity_CPC2_data.v1_1.tgz"

with tarfile.open(archive_path, "r:gz") as archive:
    for member in archive:
        if "metadata" in Path(member.name).parts and member.isfile():
            archive.extract(member, path=project_dir, filter="data")

print("Metadata extraction complete.")


EOFError: Compressed file ended before the end-of-stream marker was reached

In [11]:
from google.colab import files

uploaded = files.upload()

Saving cpc2_metadata.zip to cpc2_metadata (1).zip


In [12]:
from pathlib import Path
import zipfile
import json
import pandas as pd

metadata_dir = Path("/content/cpc2_metadata")
metadata_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile("cpc2_metadata.zip") as archive:
    archive.extractall(metadata_dir)

with open(metadata_dir / "CEC2.train.1.json", encoding="utf-8") as file:
    responses = pd.DataFrame(json.load(file))

with open(metadata_dir / "listeners.json", encoding="utf-8") as file:
    listeners = json.load(file)

print("Ready!")
print("Response rows:", len(responses))
print("Listeners:", len(listeners))

Ready!
Response rows: 2779
Listeners: 49


In [13]:
listener_files = list(project_dir.rglob("listeners.json"))

if not listener_files:
    raise FileNotFoundError("listeners.json was not found after extraction.")

metadata_dir = listener_files[0].parent

print("Metadata folder:", metadata_dir)

for path in sorted(metadata_dir.glob("*.json")):
    print(path.name)

Metadata folder: /content/clarity_cpc2/clarity_CPC2_data/clarity_data/metadata
._listeners.json
CEC1.train.1.json
CEC1.train.2.json
CEC1.train.3.json
CEC2.train.1.json
CEC2.train.2.json
CEC2.train.3.json
listeners.json
scenes.CEC1.json
scenes.CEC2.json


In [14]:
import json
import pandas as pd

response_path = metadata_dir / "CEC2.train.1.json"

with open(response_path, encoding="utf-8") as file:
    records = json.load(file)

responses = pd.DataFrame(records)

print("Rows and columns:", responses.shape)
display(responses.head())

Rows and columns: (2779, 10)


,prompt,scene,n_words,hits,listener,system,response,volume,signal,correctness
0,i could feel its presence and now i can't,S08792,9,5,L0200,E008_hr,i could feel and now,96,S08792_L0200_E008_hr,55.555556
1,the engineer would go to sleep soon in a corner,S09161,10,2,L0200,E008_hr,their engineer was in at,96,S09161_L0200_E008_hr,20.000000
2,and to ask you to support dad,S09377,7,1,L0200,E008_hr,to cook over,96,S09377_L0200_E008_hr,14.285714
3,i was about to telephone the police,S08892,7,2,L0200,E008_hr,to telephone,96,S08892_L0200_E008_hr,28.571429
4,finally he put it in his mouth,S09432,7,0,L0200,E008_hr,,96,S09432_L0200_E008_hr,0.000000


In [15]:
import json

with open(metadata_dir / "listeners.json", encoding="utf-8") as file:
    listeners = json.load(file)

print("Number of listeners:", len(listeners))

Number of listeners: 49


In [16]:
first_response = responses.iloc[0]
listener_id = first_response["listener"]

print("Listener ID:", listener_id)
print(listeners[listener_id])

Listener ID: L0200
{'name': 'L0200', 'audiogram_cfs': [250, 500, 1000, 2000, 3000, 4000, 6000, 8000], 'audiogram_levels_l': [35, 30, 25, 50, 55, 65, 70, 65], 'audiogram_levels_r': [45, 45, 20, 50, 70, 65, 80, 75]}


CONNECTING THE AUDIOGRAM WITH THE PERSON


In [17]:
profile = listeners[listener_id]

audiogram = pd.DataFrame({
    "Frequency (Hz)": profile["audiogram_cfs"],
    "Left ear (dB HL)": profile["audiogram_levels_l"],
    "Right ear (dB HL)": profile["audiogram_levels_r"]
})

display(audiogram)

,Frequency (Hz),Left ear (dB HL),Right ear (dB HL)
0,250,35,45
1,500,30,45
2,1000,25,20
3,2000,50,50
4,3000,55,70
5,4000,65,65
6,6000,70,80
7,8000,65,75


In [19]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=audiogram)

https://docs.google.com/spreadsheets/d/1CNRe6zoT02kJXkOqiFzM6J3aDaOTVd_NbQaO-yL8bBY/edit#gid=0


In [9]:
# from pathlib import Path

# matches = list(Path("/content/clarity_cpc2").rglob("listeners.json"))
# print(matches)

[PosixPath('/content/clarity_cpc2/clarity_CPC2_data/clarity_data/metadata/listeners.json')]


In [10]:
# import shutil
# from google.colab import files

# metadata_dir = matches[0].parent

# backup_path = shutil.make_archive(
#     "/content/cpc2_metadata",
#     "zip",
#     root_dir=metadata_dir
# )

# files.download(backup_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
# Select the first listening-test record
example = responses.iloc[0]

# Find the corresponding listener's hearing profile
listener_id = example["listener"]
profile = listeners[listener_id]

print("Listener:", listener_id)
print("Original sentence:", example["prompt"])
print("Listener's response:", example["response"])
print(f"Words understood: {example['correctness']:.1f}%")

# Display hearing measurements for both ears
audiogram = pd.DataFrame({
    "Frequency (Hz)": profile["audiogram_cfs"],
    "Left ear (dB HL)": profile["audiogram_levels_l"],
    "Right ear (dB HL)": profile["audiogram_levels_r"]
})

display(audiogram)

Listener: L0200
Original sentence: i could feel its presence and now i can't
Listener's response: i could feel and now
Words understood: 55.6%


,Frequency (Hz),Left ear (dB HL),Right ear (dB HL)
0,250,35,45
1,500,30,45
2,1000,25,20
3,2000,50,50
4,3000,55,70
5,4000,65,65
6,6000,70,80
7,8000,65,75


In [22]:
listener_rows = []

for listener_id, profile in listeners.items():
    row = {"listener": listener_id}

    for frequency, left, right in zip(
        profile["audiogram_cfs"],
        profile["audiogram_levels_l"],
        profile["audiogram_levels_r"]
    ):
        frequency = int(frequency)

        row[f"left_{frequency}_Hz"] = left
        row[f"right_{frequency}_Hz"] = right

    listener_rows.append(row)

listener_features = pd.DataFrame(listener_rows)

display(listener_features.head(20))

,listener,left_250_Hz,right_250_Hz,left_500_Hz,right_500_Hz,left_1000_Hz,right_1000_Hz,left_2000_Hz,right_2000_Hz,left_3000_Hz,right_3000_Hz,left_4000_Hz,right_4000_Hz,left_6000_Hz,right_6000_Hz,left_8000_Hz,right_8000_Hz
0,L0200,35,45,30,45,25,20,50,50,55,70,65,65,70,80,65,75
1,L0201,10,5,10,5,20,10,60,30,85,40,80,60,85,85,75,85
2,L0202,25,30,35,35,50,50,45,40,50,45,45,50,65,50,70,65
3,L0203,45,45,45,45,50,45,55,50,55,55,55,55,55,70,65,65
4,L0204,15,10,20,15,25,20,40,40,40,50,50,50,50,35,40,40
5,L0205,30,25,30,25,35,40,45,40,60,50,75,75,70,80,80,95
6,L0206,10,30,10,45,25,55,40,65,45,70,45,65,65,85,90,85
7,L0207,35,25,35,20,50,25,50,45,55,45,60,55,55,65,60,60
8,L0208,20,25,15,20,25,15,25,20,35,30,45,45,65,65,70,55
9,L0209,15,15,20,15,30,30,55,60,55,60,60,60,85,65,85,70


In [23]:
data = responses.merge(
    listener_features,
    on="listener",
    how="left",
    validate="many_to_one"
)

hearing_columns = [
    column for column in listener_features.columns
    if column != "listener"
]

display(data[["listener", "correctness"] + hearing_columns].head())

print("Number of listening-test examples:", len(data))
print("Number of hearing features:", len(hearing_columns))
print("Missing hearing measurements:", data[hearing_columns].isna().sum().sum())

,listener,correctness,left_250_Hz,right_250_Hz,left_500_Hz,right_500_Hz,left_1000_Hz,right_1000_Hz,left_2000_Hz,right_2000_Hz,left_3000_Hz,right_3000_Hz,left_4000_Hz,right_4000_Hz,left_6000_Hz,right_6000_Hz,left_8000_Hz,right_8000_Hz
0,L0200,55.555556,35,45,30,45,25,20,50,50,55,70,65,65,70,80,65,75
1,L0200,20.000000,35,45,30,45,25,20,50,50,55,70,65,65,70,80,65,75
2,L0200,14.285714,35,45,30,45,25,20,50,50,55,70,65,65,70,80,65,75
3,L0200,28.571429,35,45,30,45,25,20,50,50,55,70,65,65,70,80,65,75
4,L0200,0.000000,35,45,30,45,25,20,50,50,55,70,65,65,70,80,65,75


Number of listening-test examples: 2779
Number of hearing features: 16
Missing hearing measurements: 0


SPLITTING BY PEOPLE SO THAT THEY DONT APPEAR IN DIFFERENT GROUPS

In [25]:
from sklearn.model_selection import GroupShuffleSplit

X = data[hearing_columns]
y = data["correctness"]
groups = data["listener"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

In [26]:
train_listeners = set(groups.iloc[train_idx])
val_listeners = set(groups.iloc[val_idx])

print("Training listeners:", len(train_listeners))
print("Validation listeners:", len(val_listeners))
print("Listeners in both:", train_listeners & val_listeners)
print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))

Training listeners: 8
Validation listeners: 2
Listeners in both: set()
Training examples: 2221
Validation examples: 558


ESTABLISHING A SIMPLE BASELINE (TRAINING MEAN INTERLLIGIBILIIITY)

In [27]:
import numpy as np

baseline_score = y_train.mean()
baseline_predictions = np.full(len(y_val), baseline_score)

baseline_rmse = np.sqrt(
    np.mean((y_val.to_numpy() - baseline_predictions) ** 2)
)

print(f"Constant prediction: {baseline_score:.1f}% understood")
print(f"Baseline validation RMSE: {baseline_rmse:.2f} percentage points")

Constant prediction: 54.1% understood
Baseline validation RMSE: 39.90 percentage points


TRAINING THE MODEL

In [28]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,
    random_state=42
)

forest.fit(X_train, y_train)

RandomForestRegressor(max_depth=3, random_state=42)

In [29]:
train_predictions = forest.predict(X_train)
val_predictions = forest.predict(X_val)

train_rmse = np.sqrt(
    np.mean((y_train.to_numpy() - train_predictions) ** 2)
)

val_rmse = np.sqrt(
    np.mean((y_val.to_numpy() - val_predictions) ** 2)
)

print(f"Training RMSE: {train_rmse:.2f} percentage points")
print(f"Validation RMSE: {val_rmse:.2f} percentage points")
print(f"Baseline validation RMSE: {baseline_rmse:.2f} percentage points")

Training RMSE: 39.46 percentage points
Validation RMSE: 39.79 percentage points
Baseline validation RMSE: 39.90 percentage points


In [30]:
from pathlib import Path

project_dir = Path("/content/clarity_cpc2")

first_wav = next(project_dir.rglob("*.wav"), None)

if first_wav is None:
    print("No extracted audio found.")
else:
    print("Example audio file:", first_wav)

archive_path = project_dir / "clarity_CPC2_data.v1_1.tgz"
print("Dataset archive available:", archive_path.exists())

No extracted audio found.
Dataset archive available: True
